In [ ]:
import cv2
import numpy as np
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.feature import blob_log  # type: ignore

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def show_image(img, title="Image", skin_mask=None):
    """
    Displays an image using Matplotlib with optional skin mask application.
    """
    plt.figure(figsize=(3, 3), dpi=600)
    
    if skin_mask is not None:
        img = cv2.bitwise_and(img, img, mask=skin_mask)
        
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
    plt.title(title, fontsize=9, fontweight='light')
    plt.axis('off')
    plt.show()

# Set up project directory paths dynamically
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent
SRC_DIR = CURRENT_DIR / "src"
SHARED_DIR = PROJECT_ROOT / "shared"
YOLO_DIR = PROJECT_ROOT / "Models" / "yolo"

for path_dir in [SRC_DIR, SHARED_DIR, YOLO_DIR]:
    if str(path_dir) not in sys.path:
        sys.path.insert(0, str(path_dir))

IMAGES_DIR = PROJECT_ROOT / "images" / "zuza" / "inzynierka" / "IMG_20251129_093304954.jpg"

image = cv2.imread(str(IMAGES_DIR))
if image is None:
    raise FileNotFoundError(f"Could not load image from:\n{IMAGES_DIR}")

show_image(image, title="Original Image")

def create_landmarker(model_path: Path) -> vision.FaceLandmarker:
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(options)

FACE_OVAL_INDICES = [
    10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
    397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109
]

def crop_face_with_mediapipe(image, model_path, top_head_expansion=0.3, chin_padding=0.1, 
                           side_padding=0.08, target_aspect_ratio=0.75):
    if not model_path.exists():
        print(f"Warning: MediaPipe model does not exist at {model_path}. Skipping crop.")
        return image

    h, w, _ = image.shape
    detector = create_landmarker(model_path)

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    detection_result = detector.detect(mp_image)

    if not detection_result.face_landmarks:
        print("No face detected by MediaPipe. Using full image.")
        return image

    landmarks = detection_result.face_landmarks[0]

    img_visualization = image.copy()
    num_pts = len(FACE_OVAL_INDICES)
    for i in range(num_pts):
        idx1 = FACE_OVAL_INDICES[i]
        idx2 = FACE_OVAL_INDICES[(i + 1) % num_pts]
        pt1 = (int(landmarks[idx1].x * w), int(landmarks[idx1].y * h))
        pt2 = (int(landmarks[idx2].x * w), int(landmarks[idx2].y * h))
        cv2.line(img_visualization, pt1, pt2, (0, 255, 255), 2)

    nose_lm, left_lm, right_lm = landmarks[1], landmarks[234], landmarks[454]
    pt_nose = (int(nose_lm.x * w), int(nose_lm.y * h))
    pt_left = (int(left_lm.x * w), int(left_lm.y * h))
    pt_right = (int(right_lm.x * w), int(right_lm.y * h))

    cv2.line(img_visualization, pt_left, pt_right, (255, 255, 0), 1)
    cv2.circle(img_visualization, pt_left, 6, (255, 0, 0), -1)    
    cv2.circle(img_visualization, pt_right, 6, (0, 0, 255), -1)   
    cv2.circle(img_visualization, pt_nose, 8, (0, 255, 0), -1)    
    
    # show_image(img_visualization, "Face Landmarks")

    x_coords = [lm.x * w for lm in landmarks]
    y_coords = [lm.y * h for lm in landmarks]

    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    face_w = x_max - x_min
    face_h = y_max - y_min

    crop_x1 = x_min - (face_w * side_padding)
    crop_x2 = x_max + (face_w * side_padding)
    crop_y1 = y_min - (face_h * top_head_expansion)  
    crop_y2 = y_max + (face_h * chin_padding)        

    crop_w = crop_x2 - crop_x1
    crop_h = crop_y2 - crop_y1

    current_ratio = crop_w / crop_h

    if current_ratio < target_aspect_ratio:
        new_w = crop_h * target_aspect_ratio
        diff_w = new_w - crop_w
        crop_x1 -= diff_w / 2
        crop_x2 += diff_w / 2
    else:
        new_h = crop_w / target_aspect_ratio
        diff_h = new_h - crop_h
        crop_y1 -= diff_h * 0.7  
        crop_y2 += diff_h * 0.3  

    x1 = int(max(0, crop_x1))
    x2 = int(min(w, crop_x2))
    y1 = int(max(0, crop_y1))
    y2 = int(min(h, crop_y2))

    cropped_face = image[y1:y2, x1:x2]
    return cropped_face, img_visualization

MODEL_PATH = Path("face_landmarker.task")
cropped_image, img_visualization = crop_face_with_mediapipe(image, MODEL_PATH)
show_image(img_visualization, "Face Landmarks")
show_image(cropped_image, "Cropped Face", skin_mask=None)

SKIN_H_MIN = 0
SKIN_H_MAX = 30      
SKIN_S_MIN = 15      
SKIN_S_MAX = 255     
SKIN_V_MIN = 40      
SKIN_CR_MIN = 130    
SKIN_CR_MAX = 190    
SKIN_CB_MIN = 70     
SKIN_CB_MAX = 140    

REF_DIAG = 1414.0    
BASE_BLUR = 5        
BASE_ADAPTIVE_BLOCK = 31 
BASE_MORPH_KERNEL = 5   

def get_adaptive_params(image):
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    scale = diag / REF_DIAG
    blur_k = max(3, int(BASE_BLUR * scale) | 1)
    adaptive_block = max(5, int(BASE_ADAPTIVE_BLOCK * scale) | 1)
    k_close = max(3, int(diag * 0.003) | 1)
    k_open = max(3, int(diag * 0.0015) | 1)
    return blur_k, adaptive_block, k_close, k_open

def create_skin_mask_exp_a(image):
    h, w = image.shape[:2]
    blur_k, adaptive_block, k_close, k_open = get_adaptive_params(image)
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv_mask = cv2.inRange(
        hsv,
        np.array([SKIN_H_MIN, SKIN_S_MIN, SKIN_V_MIN]),
        np.array([SKIN_H_MAX, SKIN_S_MAX, 255])
    )

    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    _, cr, cb = cv2.split(ycrcb)

    cr_mask = cv2.inRange(cr, SKIN_CR_MIN, SKIN_CR_MAX)
    cb_mask = cv2.inRange(cb, SKIN_CB_MIN, SKIN_CB_MAX)
    ycrcb_mask = cv2.bitwise_and(cr_mask, cb_mask)

    skin_mask = cv2.bitwise_or(hsv_mask, ycrcb_mask)

    kernel_close = np.ones((k_close, k_close), np.uint8)
    kernel_open = np.ones((k_open, k_open), np.uint8)
    
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel_close, iterations=2)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)

    return skin_mask

skin_mask = create_skin_mask_exp_a(cropped_image)

def create_acne_heatmap(image, skin_mask):
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    scale = diag / 1414.0
    bilateral_d = max(3, int(9 * scale) | 1)
    blur_ksize = max(3, int(15 * scale) | 1)

    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    _, a_channel, _ = cv2.split(lab)
    
    a_denoised = cv2.bilateralFilter(a_channel, d=bilateral_d, sigmaColor=75, sigmaSpace=75)
    
    a_skin_pixels = a_denoised[skin_mask > 0]
    if len(a_skin_pixels) == 0:
        return np.zeros_like(image)
    
    median_skin_redness = np.median(a_skin_pixels)
    a_diff = np.maximum(0, a_denoised.astype(np.float32) - median_skin_redness)
    a_diff[skin_mask == 0] = 0
    
    non_zero_diffs = a_diff[a_diff > 0]
    if len(non_zero_diffs) > 0:
        p99 = np.percentile(non_zero_diffs, 98)
        a_clipped = np.clip(a_diff, 0, p99)
        redness_norm = cv2.normalize(a_clipped, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX).astype(np.uint8)
    else:
        redness_norm = np.zeros_like(a_denoised, dtype=np.uint8)
    
    redness_smooth = cv2.GaussianBlur(redness_norm, (blur_ksize, blur_ksize), 0)
    heatmap_color = cv2.applyColorMap(redness_smooth, cv2.COLORMAP_JET)
    heatmap_color[skin_mask == 0] = 0
    
    return heatmap_color

perfect_heatmap = create_acne_heatmap(cropped_image, skin_mask)
show_image(perfect_heatmap, "Heatmap")

def enhance_details(img, clip_l=2.0, clip_a=3.0, grid_size=(8, 8), blur_kernel=5):
    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe_l = cv2.createCLAHE(clipLimit=clip_l, tileGridSize=grid_size)
    l_enhanced = clahe_l.apply(l)
    
    clahe_a = cv2.createCLAHE(clipLimit=clip_a, tileGridSize=grid_size)
    a_enhanced = clahe_a.apply(a)
    
    redness_map = cv2.subtract(a_enhanced, 128)
    redness_map = cv2.normalize(redness_map, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    
    a_enhanced = cv2.medianBlur(a_enhanced, blur_kernel)
    enhanced_img = cv2.merge((l_enhanced, a_enhanced, b))
    enhanced_img = cv2.cvtColor(enhanced_img, cv2.COLOR_LAB2BGR)
    
    return enhanced_img, redness_map

enhanced_image, redness_map = enhance_details(cropped_image, clip_l=1.2, clip_a=1.1, grid_size=(8, 8))
skin_mask_enhanced = create_skin_mask_exp_a(enhanced_image)
perfect_heatmap_enhanced = create_acne_heatmap(enhanced_image, skin_mask_enhanced)

def create_base_mask_from_heatmap(perfect_heatmap, skin_mask, percentile=85):
    """
    Poprawiona maska bazowa:
    1. Czerwone punkty (najwyższe wartości w skali szarości heatmapy) wchodzą BEZWZGLĘDNIE na stałe.
    2. Obszary pośrednie (żółte) wchodzą tylko wtedy, gdy towarzyszy im nagły kontrast krawędzi.
    """
    gray = cv2.cvtColor(perfect_heatmap, cv2.COLOR_BGR2GRAY)
    skin_pixels_gray = gray[skin_mask > 0]
    
    if len(skin_pixels_gray) == 0:
        return np.zeros_like(gray)
        
    # 1. Bezwzględne wpuszczenie najsilniejszych (czerwonych) zmian bez dziur
    p_val = np.percentile(skin_pixels_gray, percentile)
    _, permanent_red_mask = cv2.threshold(gray, p_val, 255, cv2.THRESH_BINARY)
    
    # Domknięcie ewentualnych dziur wewnątrz mocnych czerwonych plam
    h, w = gray.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    k_size = max(3, int(diag * 0.002) | 1)
    kernel = np.ones((k_size, k_size), np.uint8)
    permanent_red_mask = cv2.morphologyEx(permanent_red_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # 2. Wykrywanie kontrastu dla obszarów pośrednich (żółtych)
    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 1, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gradient_magnitude = cv2.magnitude(sobel_x, sobel_y)
    gradient_norm = cv2.normalize(gradient_magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    _, contrast_mask = cv2.threshold(gradient_norm, 35, 255, cv2.THRESH_BINARY)
    _, yellow_potential = cv2.threshold(gray, p_val * 0.70, 255, cv2.THRESH_BINARY)
    yellow_with_contrast = cv2.bitwise_and(yellow_potential, contrast_mask)
    
    # 3. Połączenie warstw
    base_mask = cv2.bitwise_or(permanent_red_mask, yellow_with_contrast)
    base_mask = cv2.bitwise_and(base_mask, skin_mask)
    
    return base_mask

base_mask_org = create_base_mask_from_heatmap(perfect_heatmap, skin_mask, percentile=90)
show_image(base_mask_org, "Base Mask (Original Image)", skin_mask=skin_mask)

base_mask_enhanced = create_base_mask_from_heatmap(perfect_heatmap_enhanced, skin_mask_enhanced, percentile=85)
show_image(base_mask_enhanced, "Base Mask (Enhanced Image)", skin_mask=skin_mask_enhanced)

def filter_mask_variations(mask, image, skin_mask, perfect_heatmap):
    h, w = mask.shape[:2]
    total_pixels = h * w
    diag = np.sqrt(h**2 + w**2)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detected_objects = []
    
    heatmap_gray = cv2.cvtColor(perfect_heatmap, cv2.COLOR_BGR2GRAY)
    ideal_radius = diag * 0.0035  
    
    obj_id = 1
    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        
        if area <= 0 or perimeter == 0:
            continue
            
        (cx, cy), radius = cv2.minEnclosingCircle(cnt)
        
        max_allowed_area = total_pixels * 0.0015 
        min_allowed_area = max(2, int(total_pixels * 0.000001))
        
        if area > max_allowed_area:
            continue  
            
        mask_single = np.zeros(mask.shape, dtype=np.uint8)
        cv2.drawContours(mask_single, [cnt], -1, 255, -1)
        
        # POPRAWKA: zmieniono > 255 na > 0 oraz użyto np.max dla czerwonych punktów
        heatmap_pixels = heatmap_gray[mask_single > 0]
        max_intensity = np.max(heatmap_pixels) if len(heatmap_pixels) > 0 else 0.0
        
        raw_intensity = min(1.0, max_intensity / 255.0) 
        intensity_score = raw_intensity ** 2.0 
        
        is_strong_red = (raw_intensity >= 0.65)
        
        if not is_strong_red and area < min_allowed_area:
            continue
            
        circularity = (4 * np.pi * area) / (perimeter ** 2)
        circularity_score = min(1.0, max(0.0, circularity))
        
        size_diff = abs(radius - ideal_radius)
        size_score = max(0.0, 1.0 - (size_diff / (ideal_radius * 2.0)))
        
        if is_strong_red:
            confidence = max(0.85, (0.7 * intensity_score) + (0.2 * circularity_score) + (0.1 * size_score))
        else:
            confidence = (0.7 * intensity_score) + (0.2 * circularity_score) + (0.1 * size_score)
        
        x, y, w_box, h_box = cv2.boundingRect(cnt)
        
        detected_objects.append({
            'id': obj_id,
            'contour': cnt,
            'confidence': float(confidence),
            'raw_intensity': float(raw_intensity),
            'intensity_score': float(intensity_score),
            'circularity_score': float(circularity_score),
            'size_score': float(size_score),
            'box': (x, y, w_box, h_box)
        })
        obj_id += 1
        
    return detected_objects

filtered_objects = filter_mask_variations(base_mask_org, cropped_image, skin_mask, perfect_heatmap)
filtered_objects_enhanced = filter_mask_variations(base_mask_enhanced, cropped_image, skin_mask_enhanced, perfect_heatmap_enhanced)

# WYŚWIETLENIE PRZEFILTROWANEJ MASKI (ZGODNIE Z PROŚBĄ)
def get_binary_mask_from_objects(mask_shape, detected_objects):
    res_mask = np.zeros(mask_shape, dtype=np.uint8)
    for obj in detected_objects:
        cv2.drawContours(res_mask, [obj['contour']], -1, 255, -1)
    return res_mask

filtered_mask_viz = get_binary_mask_from_objects(base_mask_org.shape, filtered_objects)
show_image(filtered_mask_viz, "Przefiltrowana Maska (Oryginał)", skin_mask=skin_mask)

filtered_mask_viz_enh = get_binary_mask_from_objects(base_mask_enhanced.shape, filtered_objects_enhanced)
show_image(filtered_mask_viz_enh, "Przefiltrowana Maska (Enhanced)", skin_mask=skin_mask_enhanced)

def print_debug_dataframe(detected_objects, title="Tabela debugowania wykryć"):
    if not detected_objects:
        print(f"Brak wykrytych obiektów dla: {title}")
        return
    
    df_data = []
    for obj in detected_objects:
        df_data.append({
            'ID': obj['id'],
            'Confidence': round(obj['confidence'], 3),
            'Raw Intensity': round(obj['raw_intensity'], 3),
            'Intensity Score': round(obj['intensity_score'], 3),
            'Circularity Score': round(obj['circularity_score'], 3),
            'Size Score': round(obj['size_score'], 3)
        })
    df = pd.DataFrame(df_data)
    print(f"\n--- {title} ---")
    print(df.to_string(index=False))

print_debug_dataframe(filtered_objects, "Wykrycia - Obraz Oryginalny")
print_debug_dataframe(filtered_objects_enhanced, "Wykrycia - Obraz Enhanced")

def detect_and_draw_scored_boxes(image, detected_objects, padding_factor=1.2, min_confidence=0.08):
    result_img = image.copy()
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    line_thickness = max(1, int(diag * 0.0008))
    
    ideal_radius = diag * 0.0035  
    detected_count = 0
    
    for obj in detected_objects:
        if obj['confidence'] < min_confidence:
            continue
            
        x, y, w_box, h_box = obj['box']
        conf = obj['confidence']
        obj_id = obj['id']
        cnt = obj['contour']
        
        _, radius = cv2.minEnclosingCircle(cnt)
        size_error = abs(radius - ideal_radius) / ideal_radius  
        
        cx = x + w_box // 2
        cy = y + h_box // 2
        
        object_size = max(w_box, h_box)
        safe_object_size = max(5, object_size)
        dynamic_padding = max(1.1, min(2.5, padding_factor + (10.0 / safe_object_size)))
        
        size = object_size * dynamic_padding
        half_size = int(size / 2)
        
        x1 = max(0, cx - half_size)
        y1 = max(0, cy - half_size)
        x2 = min(w, cx + half_size)
        y2 = min(h, cy + half_size)
        
        if size_error <= 0.25:
            color = (0, 255, 255)  # Żółty
        else:
            color = (0, int(255 * conf), int(255 * (1 - conf)))
        
        cv2.rectangle(result_img, (x1, y1), (x2, y2), color, line_thickness, cv2.LINE_AA)
        
        label = f"ID:{obj_id} ({conf:.2f})"
        cv2.putText(result_img, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
        
        detected_count += 1

    return result_img, detected_count

final_boxes_image, count = detect_and_draw_scored_boxes(cropped_image, filtered_objects, padding_factor=1.2, min_confidence=0.15)
print(f"Liczba wykrytych zmian (Oryginał): {count}")
show_image(final_boxes_image, "Finalne wykrycia z Confidence i ID", skin_mask=skin_mask)

final_boxes_image_enh, count_enh = detect_and_draw_scored_boxes(cropped_image, filtered_objects_enhanced, padding_factor=1.2, min_confidence=0.1)
print(f"Liczba wykrytych zmian (Enhanced): {count_enh}")
show_image(final_boxes_image_enh, "Finalne wykrycia (enhanced) z Confidence i ID", skin_mask=skin_mask_enhanced)

In [ ]:
import cv2
import numpy as np
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def show_image(img, title="Image", skin_mask=None):
    """
    Displays an image using Matplotlib with optional skin mask application.
    """
    plt.figure(figsize=(3, 3), dpi=600)
    
    if skin_mask is not None:
        img = cv2.bitwise_and(img, img, mask=skin_mask)
        
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
    plt.title(title, fontsize=9, fontweight='light')
    plt.axis('off')
    plt.show()

# Set up project directory paths dynamically
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "src" else CURRENT_DIR
SRC_DIR = CURRENT_DIR / "src"
SHARED_DIR = PROJECT_ROOT / "shared"
YOLO_DIR = PROJECT_ROOT / "Models" / "yolo"

for path_dir in [SRC_DIR, SHARED_DIR, YOLO_DIR]:
    if str(path_dir) not in sys.path:
        sys.path.insert(0, str(path_dir))

IMAGES_DIR = PROJECT_ROOT / "images" / "zuza" / "inzynierka" / "IMG_20251129_093304954.jpg"

image = cv2.imread(str(IMAGES_DIR))
if image is None:
    raise FileNotFoundError(f"Could not load image from:\n{IMAGES_DIR}")

show_image(image, title="Original Image")

def create_landmarker(model_path: Path) -> vision.FaceLandmarker:
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(options)

FACE_OVAL_INDICES = [
    10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
    397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109
]

def crop_face_with_mediapipe(image, model_path, top_head_expansion=0.3, chin_padding=0.1, 
                           side_padding=0.08, target_aspect_ratio=0.75):
    if not model_path.exists():
        print(f"Warning: MediaPipe model does not exist at {model_path}. Skipping crop.")
        return image

    h, w, _ = image.shape
    detector = create_landmarker(model_path)

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    detection_result = detector.detect(mp_image)

    if not detection_result.face_landmarks:
        print("No face detected by MediaPipe. Using full image.")
        return image

    landmarks = detection_result.face_landmarks[0]

    img_visualization = image.copy()
    num_pts = len(FACE_OVAL_INDICES)
    for i in range(num_pts):
        idx1 = FACE_OVAL_INDICES[i]
        idx2 = FACE_OVAL_INDICES[(i + 1) % num_pts]
        pt1 = (int(landmarks[idx1].x * w), int(landmarks[idx1].y * h))
        pt2 = (int(landmarks[idx2].x * w), int(landmarks[idx2].y * h))
        cv2.line(img_visualization, pt1, pt2, (0, 255, 255), 2)

    nose_lm, left_lm, right_lm = landmarks[1], landmarks[234], landmarks[454]
    pt_nose = (int(nose_lm.x * w), int(nose_lm.y * h))
    pt_left = (int(left_lm.x * w), int(left_lm.y * h))
    pt_right = (int(right_lm.x * w), int(right_lm.y * h))

    cv2.line(img_visualization, pt_left, pt_right, (255, 255, 0), 1)
    cv2.circle(img_visualization, pt_left, 6, (255, 0, 0), -1)    
    cv2.circle(img_visualization, pt_right, 6, (0, 0, 255), -1)   
    cv2.circle(img_visualization, pt_nose, 8, (0, 255, 0), -1)    
    
    # show_image(img_visualization, "Face Landmarks")

    x_coords = [lm.x * w for lm in landmarks]
    y_coords = [lm.y * h for lm in landmarks]

    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    face_w = x_max - x_min
    face_h = y_max - y_min

    crop_x1 = x_min - (face_w * side_padding)
    crop_x2 = x_max + (face_w * side_padding)
    crop_y1 = y_min - (face_h * top_head_expansion)  
    crop_y2 = y_max + (face_h * chin_padding)        

    crop_w = crop_x2 - crop_x1
    crop_h = crop_y2 - crop_y1

    current_ratio = crop_w / crop_h

    if current_ratio < target_aspect_ratio:
        new_w = crop_h * target_aspect_ratio
        diff_w = new_w - crop_w
        crop_x1 -= diff_w / 2
        crop_x2 += diff_w / 2
    else:
        new_h = crop_w / target_aspect_ratio
        diff_h = new_h - crop_h
        crop_y1 -= diff_h * 0.7  
        crop_y2 += diff_h * 0.3  

    x1 = int(max(0, crop_x1))
    x2 = int(min(w, crop_x2))
    y1 = int(max(0, crop_y1))
    y2 = int(min(h, crop_y2))

    cropped_face = image[y1:y2, x1:x2]
    return cropped_face, img_visualization

MODEL_PATH = Path("face_landmarker.task")
cropped_image, img_visualization = crop_face_with_mediapipe(image, MODEL_PATH)
show_image(img_visualization, "Face Landmarks")
show_image(cropped_image, "Cropped Face", skin_mask=None)

SKIN_H_MIN = 0
SKIN_H_MAX = 30      
SKIN_S_MIN = 15      
SKIN_S_MAX = 255     
SKIN_V_MIN = 40      
SKIN_CR_MIN = 130    
SKIN_CR_MAX = 190    
SKIN_CB_MIN = 70     
SKIN_CB_MAX = 140    

REF_DIAG = 1414.0    
BASE_BLUR = 5        
BASE_ADAPTIVE_BLOCK = 31 
BASE_MORPH_KERNEL = 5   

def get_adaptive_params(image):
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    scale = diag / REF_DIAG
    blur_k = max(3, int(BASE_BLUR * scale) | 1)
    adaptive_block = max(5, int(BASE_ADAPTIVE_BLOCK * scale) | 1)
    k_close = max(3, int(diag * 0.003) | 1)
    k_open = max(3, int(diag * 0.0015) | 1)
    return blur_k, adaptive_block, k_close, k_open

def create_skin_mask_exp_a(image):
    h, w = image.shape[:2]
    blur_k, adaptive_block, k_close, k_open = get_adaptive_params(image)
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv_mask = cv2.inRange(
        hsv,
        np.array([SKIN_H_MIN, SKIN_S_MIN, SKIN_V_MIN]),
        np.array([SKIN_H_MAX, SKIN_S_MAX, 255])
    )

    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    _, cr, cb = cv2.split(ycrcb)

    cr_mask = cv2.inRange(cr, SKIN_CR_MIN, SKIN_CR_MAX)
    cb_mask = cv2.inRange(cb, SKIN_CB_MIN, SKIN_CB_MAX)
    ycrcb_mask = cv2.bitwise_and(cr_mask, cb_mask)

    skin_mask = cv2.bitwise_or(hsv_mask, ycrcb_mask)

    kernel_close = np.ones((k_close, k_close), np.uint8)
    kernel_open = np.ones((k_open, k_open), np.uint8)
    
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel_close, iterations=2)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)

    return skin_mask

skin_mask = create_skin_mask_exp_a(cropped_image)

def create_acne_heatmap(image, skin_mask):
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    scale = diag / 1414.0
    bilateral_d = max(3, int(9 * scale) | 1)
    blur_ksize = max(3, int(15 * scale) | 1)

    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    _, a_channel, _ = cv2.split(lab)
    
    a_denoised = cv2.bilateralFilter(a_channel, d=bilateral_d, sigmaColor=75, sigmaSpace=75)
    
    a_skin_pixels = a_denoised[skin_mask > 0]
    if len(a_skin_pixels) == 0:
        return np.zeros_like(image)
    
    median_skin_redness = np.median(a_skin_pixels)
    a_diff = np.maximum(0, a_denoised.astype(np.float32) - median_skin_redness)
    a_diff[skin_mask == 0] = 0
    
    non_zero_diffs = a_diff[a_diff > 0]
    if len(non_zero_diffs) > 0:
        p99 = np.percentile(non_zero_diffs, 98)
        a_clipped = np.clip(a_diff, 0, p99)
        redness_norm = cv2.normalize(a_clipped, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX).astype(np.uint8)
    else:
        redness_norm = np.zeros_like(a_denoised, dtype=np.uint8)
    
    redness_smooth = cv2.GaussianBlur(redness_norm, (blur_ksize, blur_ksize), 0)
    heatmap_color = cv2.applyColorMap(redness_smooth, cv2.COLORMAP_JET)
    heatmap_color[skin_mask == 0] = 0
    
    return heatmap_color

perfect_heatmap = create_acne_heatmap(cropped_image, skin_mask)
show_image(perfect_heatmap, "Heatmap")

def enhance_details(img, clip_l=2.0, clip_a=3.0, grid_size=(8, 8), blur_kernel=5):
    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe_l = cv2.createCLAHE(clipLimit=clip_l, tileGridSize=grid_size)
    l_enhanced = clahe_l.apply(l)
    
    clahe_a = cv2.createCLAHE(clipLimit=clip_a, tileGridSize=grid_size)
    a_enhanced = clahe_a.apply(a)
    
    redness_map = cv2.subtract(a_enhanced, 128)
    redness_map = cv2.normalize(redness_map, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    
    a_enhanced = cv2.medianBlur(a_enhanced, blur_kernel)
    enhanced_img = cv2.merge((l_enhanced, a_enhanced, b))
    enhanced_img = cv2.cvtColor(enhanced_img, cv2.COLOR_LAB2BGR)
    
    return enhanced_img, redness_map

enhanced_image, redness_map = enhance_details(cropped_image, clip_l=1.2, clip_a=1.1, grid_size=(8, 8))
skin_mask_enhanced = create_skin_mask_exp_a(enhanced_image)
perfect_heatmap_enhanced = create_acne_heatmap(enhanced_image, skin_mask_enhanced)
show_image(perfect_heatmap_enhanced, "Heatmap (enhanced)")

def create_base_mask_from_heatmap(perfect_heatmap, skin_mask):
    gray = cv2.cvtColor(perfect_heatmap, cv2.COLOR_BGR2GRAY)
    
    # 1. Bezwzględne zachowanie wszystkich czerwonych pikseli (próg >= 150)
    _, permanent_red_mask = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    
    # 2. Obszary pośrednie wsparte kontrastem krawędzi
    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 1, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gradient_norm = cv2.normalize(cv2.magnitude(sobel_x, sobel_y), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    _, contrast_mask = cv2.threshold(gradient_norm, 35, 255, cv2.THRESH_BINARY)
    _, yellow_potential = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
    yellow_with_contrast = cv2.bitwise_and(yellow_potential, contrast_mask)
    
    # Łączymy bazę
    base_mask = cv2.bitwise_or(permanent_red_mask, yellow_with_contrast)
    base_mask = cv2.bitwise_and(base_mask, skin_mask)
    
    # 3. FILTROWANIE TYLKO DLA ŻÓŁTYCH/SŁABSZYCH OBSZARÓW - CZERWONE ZOSTAWIAMY ZAWSZE
    contours, _ = cv2.findContours(base_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = base_mask.shape[:2]
    total_area = h * w
    
    filtered_base_mask = np.zeros_like(base_mask)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 2:
            continue
            
        # Sprawdzamy intensywność konturu
        mask_single = np.zeros(base_mask.shape, dtype=np.uint8)
        cv2.drawContours(mask_single, [cnt], -1, 255, -1)
        max_val = np.max(gray[mask_single > 0])
        
        if max_val >= 150:
            # === ZASADA: CZERWONE PUNKTY / OGNISKA PRZECHODZĄ ZAWSZE BEZ ŻADNYCH LIMITÓW ===
            cv2.drawContours(filtered_base_mask, [cnt], -1, 255, -1)
        else:
            # Tylko dla słabszych (żółtych) obszarów stosujemy restrykcje wielkości
            if area <= (total_area * 0.0015):
                cv2.drawContours(filtered_base_mask, [cnt], -1, 255, -1)
            
    return filtered_base_mask

base_mask_org = create_base_mask_from_heatmap(perfect_heatmap, skin_mask)
show_image(base_mask_org, "Przefiltrowana Maska (Oryginał)", skin_mask=skin_mask)

base_mask_enhanced = create_base_mask_from_heatmap(perfect_heatmap_enhanced, skin_mask_enhanced)
show_image(base_mask_enhanced, "Przefiltrowana Maska (Enhanced)", skin_mask=skin_mask_enhanced)

def filter_mask_variations(mask, perfect_heatmap):
    """
    Zbiera obiekty bezpośrednio z maski pikselowej i oblicza ich confidence bazując na maksimum pikseli.
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detected_objects = []
    heatmap_gray = cv2.cvtColor(perfect_heatmap, cv2.COLOR_BGR2GRAY)
    
    obj_id = 1
    for cnt in contours:
        mask_single = np.zeros(mask.shape, dtype=np.uint8)
        cv2.drawContours(mask_single, [cnt], -1, 255, -1)
        
        pixel_values = heatmap_gray[mask_single > 0]
        if len(pixel_values) == 0:
            continue
            
        max_intensity = np.max(pixel_values)
        raw_intensity = min(1.0, max_intensity / 255.0)
        intensity_score = raw_intensity ** 2.0
        
        # Prawdziwa czerwień ma wysoki confidence i przechodzi bezwzględnie
        if raw_intensity >= 0.65:
            confidence = max(0.85, intensity_score)
        else:
            confidence = intensity_score
            
        detected_objects.append({
            'id': obj_id,
            'contour': cnt,
            'confidence': float(confidence),
            'raw_intensity': float(raw_intensity),
            'intensity_score': float(intensity_score)
        })
        obj_id += 1
        
    return detected_objects

filtered_objects = filter_mask_variations(base_mask_org, perfect_heatmap)
filtered_objects_enhanced = filter_mask_variations(base_mask_enhanced, perfect_heatmap_enhanced)

# Tabela debugowania Pandas
def print_debug_dataframe(detected_objects, title="Tabela debugowania"):
    if not detected_objects:
        print(f"Brak obiektów dla: {title}")
        return
    df = pd.DataFrame([{
        'ID': o['id'], 'Confidence': round(o['confidence'], 3),
        'Raw Intensity': round(o['raw_intensity'], 3), 'Intensity Score': round(o['intensity_score'], 3)
    } for o in detected_objects])
    print(f"\n--- {title} ---")
    print(df.to_string(index=False))

print_debug_dataframe(filtered_objects, "Wykrycia - Oryginał")
print_debug_dataframe(filtered_objects_enhanced, "Wykrycia - Enhanced")

def draw_contours_on_image(image, detected_objects, min_confidence=0.08):
    """
    Zamiast ramek rysuje dokładny OBRYS (kontur) wykrytej zmiany na obrazie.
    """
    result_img = image.copy()
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    line_thickness = max(1, int(diag * 0.001))
    
    for obj in detected_objects:
        if obj['confidence'] < min_confidence:
            continue
            
        cnt = obj['contour']
        conf = obj['confidence']
        obj_id = obj['id']
        
        # Kolor dopasowany do confidence (od zielonego/żółtego do czerwonego)
        color = (0, int(255 * (1 - conf)), int(255 * conf))
        
        # Rysowanie dokładnego obrysowania wokół maski trądziku
        cv2.drawContours(result_img, [cnt], -1, color, line_thickness, cv2.LINE_AA)
        
        # Opcjonalna mała etykietka z ID i confidence przy obrysie
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
            cv2.putText(result_img, f"#{obj_id}", (cX - 10, cY), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1, cv2.LINE_AA)

    return result_img

# Rysowanie finalnych obrysów
final_contours_org = draw_contours_on_image(cropped_image, filtered_objects, min_confidence=0.15)
show_image(final_contours_org, "Finalne obrysy zmian (Oryginał)")

final_contours_enh = draw_contours_on_image(cropped_image, filtered_objects_enhanced, min_confidence=0.1)
show_image(final_contours_enh, "Finalne obrysy zmian (Enhanced)")

RED_ZONE_THRESHOLD = 150      # Od tej wartości zaczyna się strefa czerwona (najwyższe confidence)
ORANGE_ZONE_THRESHOLD = 120   # Od tej wartości strefa pomarańczowa
YELLOW_ZONE_THRESHOLD = 90    # Od tej wartości strefa żółta

def show_zone_colorbar_axis():
    gradient = np.linspace(0, 255, 500, dtype=np.uint8)
    gradient_2d = np.tile(gradient, (40, 1))
    jet_scale = cv2.applyColorMap(gradient_2d, cv2.COLORMAP_JET)
    jet_rgb = cv2.cvtColor(jet_scale, cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(figsize=(8, 2.5), dpi=300)
    ax.imshow(jet_rgb, aspect='auto', extent=[0, 255, 0, 1])
    ax.set_title("Oś Kolorystyczna Stref (Progi Globalne)", fontsize=10, fontweight='bold')
    ax.set_yticks([])
    ax.set_xlabel("Wartość piksela na mapie ciepła (0 - 255)", fontsize=9)
    
    # Linie granic stref
    ax.axvline(x=RED_ZONE_THRESHOLD, color='white', linestyle='--', linewidth=1.5)
    ax.text(RED_ZONE_THRESHOLD + 2, 0.8, f"Strefa Czerwona\n(>= {RED_ZONE_THRESHOLD})", color='white', fontsize=7, fontweight='bold', bbox=dict(facecolor='darkred', alpha=0.7))
    
    ax.axvline(x=ORANGE_ZONE_THRESHOLD, color='white', linestyle='--', linewidth=1.5)
    ax.text(ORANGE_ZONE_THRESHOLD + 2, 0.5, f"Strefa Pomarańczowa\n({ORANGE_ZONE_THRESHOLD}-{RED_ZONE_THRESHOLD})", color='black', fontsize=7, fontweight='bold', bbox=dict(facecolor='orange', alpha=0.7))
    
    ax.axvline(x=YELLOW_ZONE_THRESHOLD, color='white', linestyle='--', linewidth=1.5)
    ax.text(YELLOW_ZONE_THRESHOLD + 2, 0.2, f"Strefa Żółta\n({YELLOW_ZONE_THRESHOLD}-{ORANGE_ZONE_THRESHOLD})", color='black', fontsize=7, fontweight='bold', bbox=dict(facecolor='yellow', alpha=0.7))
    
    plt.tight_layout()
    plt.show()

show_zone_colorbar_axis()


def analyze_zones_and_draw(image, perfect_heatmap, skin_mask):
    gray_heatmap = cv2.cvtColor(perfect_heatmap, cv2.COLOR_BGR2GRAY)
    skin_pixels = gray_heatmap[skin_mask > 0]
    total_skin_count = len(skin_pixels)
    
    if total_skin_count == 0:
        return image, {}

    # Obliczamy udziały procentowe stref w skórze
    red_pixels_count = np.sum((skin_pixels >= RED_ZONE_THRESHOLD))
    orange_pixels_count = np.sum((skin_pixels >= ORANGE_ZONE_THRESHOLD) & (skin_pixels < RED_ZONE_THRESHOLD))
    yellow_pixels_count = np.sum((skin_pixels >= YELLOW_ZONE_THRESHOLD) & (skin_pixels < ORANGE_ZONE_THRESHOLD))
    blue_pixels_count = np.sum((skin_pixels < YELLOW_ZONE_THRESHOLD) & (skin_pixels > 0))

    red_pct = (red_pixels_count / total_skin_count) * 100
    orange_pct = (orange_pixels_count / total_skin_count) * 100
    yellow_pct = (yellow_pixels_count / total_skin_count) * 100
    blue_pct = (blue_pixels_count / total_skin_count) * 100

    # Logika dominacji: jeśli dominuje żółty/niebieski, ich confidence spada do 0 i nie rysujemy obrysów
    draw_blue = blue_pct < 80.0  # Jeśli niebieska to większość tła skóry, pomijamy obrys
    draw_yellow = yellow_pct < 50.0 # Jeśli żółta dominuje, confidence = 0
    draw_orange = orange_pct < 40.0
    draw_red = True # Czerwona ZAWSZE ma obrys i wysokie confidence

    result_img = image.copy()
    h, w = image.shape[:2]
    diag = np.sqrt(h**2 + w**2)
    line_thickness = max(1, int(diag * 0.0008))

    # Maski dla poszczególnych stref
    zones = {
        'Czerwona (Zapalna)': {
            'mask': cv2.bitwise_and((gray_heatmap >= RED_ZONE_THRESHOLD).astype(np.uint8) * 255, skin_mask),
            'color': (0, 0, 255), # BGR: Czerwony
            'conf': 0.95,
            'pct': red_pct,
            'active': draw_red
        },
        'Pomarańczowa': {
            'mask': cv2.bitwise_and(((gray_heatmap >= ORANGE_ZONE_THRESHOLD) & (gray_heatmap < RED_ZONE_THRESHOLD)).astype(np.uint8) * 255, skin_mask),
            'color': (0, 165, 255), # BGR: Pomarańczowy
            'conf': 0.70,
            'pct': orange_pct,
            'active': draw_orange
        },
        'Żółta': {
            'mask': cv2.bitwise_and(((gray_heatmap >= YELLOW_ZONE_THRESHOLD) & (gray_heatmap < ORANGE_ZONE_THRESHOLD)).astype(np.uint8) * 255, skin_mask),
            'color': (0, 255, 255), # BGR: Żółty
            'conf': 0.35 if draw_yellow else 0.0,
            'pct': yellow_pct,
            'active': draw_yellow
        },
        'Niebieska (Neutralna)': {
            'mask': cv2.bitwise_and(((gray_heatmap < YELLOW_ZONE_THRESHOLD) & (gray_heatmap > 0)).astype(np.uint8) * 255, skin_mask),
            'color': (255, 0, 0), # BGR: Niebieski
            'conf': 0.0,
            'pct': blue_pct,
            'active': False # Neutralna nie ma obrysu
        }
    }

    # Rysowanie obrysów wokół stref (tylko linie dookoła, brak wypełnienia)
    for zone_name, data in zones.items():
        if not data['active'] or data['conf'] == 0.0:
            continue
            
        contours, _ = cv2.findContours(data['mask'], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) > 2:
                cv2.drawContours(result_img, [cnt], -1, data['color'], line_thickness, cv2.LINE_AA)

    return result_img, zones

final_zoned_image, zone_stats = analyze_zones_and_draw(cropped_image, perfect_heatmap_enhanced, skin_mask)
show_image(final_zoned_image, "Obraz z obrysami stref")

# Wyświetlenie legendy i udziałów procentowych w konsoli
print("\n--- LEGENDA STREF I UDZIAŁ PROCENTOWY ---")
for zone_name, data in zone_stats.items():
    active_str = "Aktywna (Jest obrys)" if (data['active'] and data['conf'] > 0) else "Neutralna / Pominięta (Conf: 0)"
    print(f"Strefa: {zone_name:<22} | Udział: {data['pct']:5.2f}% | Confidence: {data['conf']:<4} | Status: {active_str}")

# BATCH

In [ ]:
# BATCH SETTINGS
import cv2
import numpy as np
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.feature import blob_log  # type: ignore

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# Set up project directory paths dynamically
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent
INPUT_PATH = PROJECT_ROOT / "images" / "zuza" / "inzynierka"

JSON_OUTPUT_DIR = PROJECT_ROOT / "images" / "zuza" /"redness_analysis"
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

In [ ]:
import json


def export_frames(detected_objects, image_shape, image_name, variant):

    h, w = image_shape[:2]
    frames = []

    for obj in detected_objects:
        contour = obj["contour"]
        x, y, bw, bh = cv2.boundingRect(contour)

        frames.append({
            "id": int(obj["id"]),
            "bbox": {
                "x": int(x),
                "y": int(y),
                "width": int(bw),
                "height": int(bh),
                "format": "xywh"
            },
            "center": {
                "x": float(x + bw / 2),
                "y": float(y + bh / 2)
            },
            "confidence": float(obj["confidence"]),
            "raw_intensity": float(obj["raw_intensity"]),
            "intensity_score": float(obj["intensity_score"])
        })

    return {
        "image": image_name,
        "variant": variant,
        "coordinate_space": "cropped_face_pixels",
        "image_width": int(w),
        "image_height": int(h),
        "count": len(frames),
        "frames": frames
    }


def extract_color_regions(heatmap, skin_mask, image_name, variant):
    """
    Wyodrębnia czerwone, pomarańczowe i żółte obszary z heatmapy JET.
    Współrzędne odnoszą się do przyciętego obrazu twarzy.
    """
    hsv = cv2.cvtColor(heatmap, cv2.COLOR_BGR2HSV)
    h_channel, s_channel, v_channel = cv2.split(hsv)

    valid = (skin_mask > 0) & (s_channel >= 80) & (v_channel >= 60)

    color_ranges = {
        "red": (
            ((h_channel <= 10) | (h_channel >= 170)) & valid
        ),
        "orange": (
            (h_channel > 10) & (h_channel <= 22) & valid
        ),
        "yellow": (
            (h_channel > 22) & (h_channel <= 35) & valid
        )
    }

    regions = []
    region_id = 1

    for color_name, condition in color_ranges.items():
        color_mask = np.where(condition, 255, 0).astype(np.uint8)

        # Łączy niewielkie przerwy w obszarach.
        kernel = np.ones((3, 3), np.uint8)
        color_mask = cv2.morphologyEx(
            color_mask, cv2.MORPH_CLOSE, kernel, iterations=1
        )

        contours, _ = cv2.findContours(
            color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            area = float(cv2.contourArea(contour))
            if area < 2:
                continue

            x, y, bw, bh = cv2.boundingRect(contour)
            moments = cv2.moments(contour)

            if moments["m00"]:
                cx = moments["m10"] / moments["m00"]
                cy = moments["m01"] / moments["m00"]
            else:
                cx = x + bw / 2
                cy = y + bh / 2

            polygon = cv2.approxPolyDP(
                contour, epsilon=0.5, closed=True
            ).reshape(-1, 2).tolist()

            region_pixels = condition[y:y + bh, x:x + bw]
            region_intensities = v_channel[y:y + bh, x:x + bw][
                region_pixels
            ]

            regions.append({
                "id": region_id,
                "color": color_name,
                "area_pixels": area,
                "bbox": {
                    "x": int(x),
                    "y": int(y),
                    "width": int(bw),
                    "height": int(bh),
                    "format": "xywh"
                },
                "center": {
                    "x": round(float(cx), 2),
                    "y": round(float(cy), 2)
                },
                "mean_hsv_value": (
                    round(float(np.mean(region_intensities)), 2)
                    if len(region_intensities) else None
                ),
                "contour": [
                    [int(point[0]), int(point[1])]
                    for point in polygon
                ]
            })
            region_id += 1

    return {
        "image": image_name,
        "variant": variant,
        "coordinate_space": "cropped_face_pixels",
        "color_definition": "HSV hue thresholds applied to JET heatmap",
        "count": len(regions),
        "regions": regions
    }


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
def process_one_image(image_path: Path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # 1. Przycięcie twarzy
    cropped, _ = crop_face_with_mediapipe(image, MODEL_PATH)

    if cropped.size == 0:
        raise ValueError("Face crop is empty.")

    # 2. Analiza oryginalnego przycięcia
    skin_mask = create_skin_mask_exp_a(cropped)
    heatmap = create_acne_heatmap(cropped, skin_mask)
    base_mask = create_base_mask_from_heatmap(heatmap, skin_mask)
    objects = filter_mask_variations(base_mask, heatmap)

    # 3. Analiza obrazu po poprawie szczegółów
    enhanced, _ = enhance_details(
        cropped,
        clip_l=1.2,
        clip_a=1.1,
        grid_size=(8, 8)
    )
    skin_mask_enhanced = create_skin_mask_exp_a(enhanced)
    heatmap_enhanced = create_acne_heatmap(
        enhanced, skin_mask_enhanced
    )
    base_mask_enhanced = create_base_mask_from_heatmap(
        heatmap_enhanced, skin_mask_enhanced
    )
    objects_enhanced = filter_mask_variations(
        base_mask_enhanced, heatmap_enhanced
    )

    # 4. Ramki — osobny JSON
    frames_data = {
        "image": image_path.name,
        "source_path": str(image_path),
        "variants": [
            export_frames(
                objects, cropped.shape, image_path.name, "original"
            ),
            export_frames(
                objects_enhanced, enhanced.shape,
                image_path.name, "enhanced"
            )
        ]
    }

    frames_path = JSON_OUTPUT_DIR / f"{image_path.stem}_frames.json"
    save_json(frames_data, frames_path)

    # 5. Obszary czerwone, pomarańczowe i żółte — osobny JSON
    regions_data = {
        "image": image_path.name,
        "source_path": str(image_path),
        "variants": [
            extract_color_regions(
                heatmap, skin_mask, image_path.name, "original"
            ),
            extract_color_regions(
                heatmap_enhanced, skin_mask_enhanced,
                image_path.name, "enhanced"
            )
        ]
    }

    regions_path = JSON_OUTPUT_DIR / f"{image_path.stem}_regions.json"
    save_json(regions_data, regions_path)

    print(
        f"OK: {image_path.name} | "
        f"ramki: {len(objects)} + {len(objects_enhanced)} | "
        f"JSON: {frames_path.name}, {regions_path.name}"
    )

    return {
        "image": image_path.name,
        "frames_original": len(objects),
        "frames_enhanced": len(objects_enhanced),
        "frames_json": str(frames_path),
        "regions_json": str(regions_path),
        "status": "ok"
    }

In [ ]:
if INPUT_PATH.is_file():
    batch_images = (
        [INPUT_PATH]
        if INPUT_PATH.suffix.lower() in VALID_EXTENSIONS
        else []
    )
elif INPUT_PATH.is_dir():
    batch_images = sorted(
        p for p in INPUT_PATH.rglob("*")
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
    )
else:
    raise FileNotFoundError(f"Input path does not exist: {INPUT_PATH}")

if not batch_images:
    raise RuntimeError(f"No images found in: {INPUT_PATH}")

print(f"Images to process: {len(batch_images)}")
print(f"JSON output: {JSON_OUTPUT_DIR}")

batch_summary = []

for index, image_path in enumerate(batch_images, start=1):
    print(f"\n[{index}/{len(batch_images)}] {image_path.name}")

    try:
        result = process_one_image(image_path)
        batch_summary.append(result)

    except Exception as error:
        print(f"ERROR: {image_path.name}: {error}")
        batch_summary.append({
            "image": image_path.name,
            "status": "error",
            "error": str(error)
        })

summary_path = JSON_OUTPUT_DIR / "batch_summary.json"
save_json(batch_summary, summary_path)

print("\nBatch completed.")
print(f"Summary: {summary_path}")
print(f"Successful: {sum(r['status'] == 'ok' for r in batch_summary)}")
print(f"Failed: {sum(r['status'] == 'error' for r in batch_summary)}")